In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# 몰수경기/추첨승 등 특수 결정 경기 크롤링

정상 박스스코어 크롤러(`KBSA_크롤링_...`)는 실제 타석 진행 기록이 있는 경기만 처리할 수 있다. 몰수경기·추첨승처럼 타석이 안 뛰어진 채 결과만 나는 경기는 원본 스코어시트 자체가 없어서 그 크롤러가 통째로 못 만들고, 그래서 season_full에서 경기 자체가 빠진다.

이 노트북은 (1) 사이트 캘린더에서 "정상 종료로 보이는"(점수가 둘 다 숫자인) game_id를 훑고, (2) 그중 우리가 원본 스코어시트로 갖고 있지 않은 것만 골라서, (3) box_score 페이지의 `.btn_result` 라벨(몰수경기 등 특수 결정 문구)과 팀/스코어/승패/대회/라운드/일시를 뽑아 `special_games_{YEAR}.xlsx`로 저장한다. 이 파일은 `스트림릿 제조.ipynb`가 자동으로 읽어서 season_full에 결과만(선수 기록 없이) 채워 넣는다.

In [ ]:
import os
import re
import time
import random
import unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup


YEAR = 2026
BASE_DIR = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
YEAR_DIR = f"{BASE_DIR}/{YEAR}"
SPECIAL_OUT = f"{BASE_DIR}/{YEAR}/season_full/special_games_{YEAR}.xlsx"
KIND_CD = 31  # 18세 이하부(고교야구)
CALENDAR_URL = "https://www.korea-baseball.com/game/calendar?kind_cd={kind_cd}&year={year}&month={month:02d}"
BOX_SCORE_URL = "https://www.korea-baseball.com/game/box_score?game_idx={game_idx}"


def fetch_calendar_completed_ids(year, month, kind_cd=KIND_CD):
    """그 달 캘린더에서 "정상 종료로 보이는"(점수가 둘 다 숫자인) game_id만 뽑는다.
    취소/예정 경기(점수가 비어있거나 숫자가 아닌 조각)는 여기서 뺀다 — 그건
    "경기 일정 크롤링.ipynb"가 따로 처리한다."""
    url = CALENDAR_URL.format(kind_cd=kind_cd, year=year, month=month)
    res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    res.encoding = "utf-8"
    soup = BeautifulSoup(res.text, "html.parser")

    ids = []
    for li in soup.select("li[date]"):
        a = li.find("a", href=re.compile(r"game_idx=\d+"))
        if not a:
            continue
        m = re.search(r"game_idx=(\d+)", a["href"])
        if not m:
            continue
        team1_p = a.find("p", class_=re.compile(r"^team1"))
        team2_p = a.find("p", class_=re.compile(r"^team2"))
        s1_el = team1_p.find("span", class_="score") if team1_p else None
        s2_el = team2_p.find("span", class_="score") if team2_p else None
        s1 = s1_el.get_text(strip=True) if s1_el else ""
        s2 = s2_el.get_text(strip=True) if s2_el else ""
        if s1.isdigit() and s2.isdigit():
            ids.append(int(m.group(1)))
    return ids


def load_known_game_ids(year_dir, year):
    """이미 원본 스코어시트(raw xlsx)로 갖고 있는 game_id 전체 — 이건 정상
    크롤링된 경기라 이 노트북의 대상이 아니다."""
    known = set()
    month_pat = re.compile(r'^\d+월$')
    file_pat = re.compile(rf'^{year}_\d{{2}}_[a-z]+\.xlsx$')
    for entry in sorted(os.listdir(year_dir)):
        entry_nfc = unicodedata.normalize('NFC', entry)
        if not month_pat.match(entry_nfc):
            continue
        month_dir = os.path.join(year_dir, entry_nfc)
        if not os.path.isdir(month_dir):
            continue
        for f in sorted(os.listdir(month_dir)):
            f_nfc = unicodedata.normalize('NFC', f)
            if not file_pat.match(f_nfc):
                continue
            try:
                df = pd.read_excel(os.path.join(month_dir, f_nfc), usecols=['game_id'])
            except Exception as e:
                print(f"  ⚠ {f_nfc} 읽기 실패: {e}")
                continue
            known.update(pd.to_numeric(df['game_id'], errors='coerce').dropna().astype(int).tolist())
    return known


def fetch_special_result(game_idx, max_retries=3, retry_wait=30):
    """box_score 페이지에서 대회명/일시/구장/라운드/양팀 이름/스코어/승패와,
    .btn_result 라벨(몰수경기 등 특수 결정 문구)을 뽑는다. .btn_result가
    없으면(=그냥 정상 경기인데 우리가 아직 못 크롤링한 것뿐일 수 있음)
    "특이사항"을 빈 값으로 둔다 — 사람이 한 번 더 확인하는 용도."""
    url = BOX_SCORE_URL.format(game_idx=game_idx)

    res = None
    for attempt in range(max_retries):
        try:
            res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
            break
        except requests.exceptions.ConnectionError as e:
            if attempt < max_retries - 1:
                print(f"  [{game_idx}] 연결 실패 ({attempt+1}/{max_retries}), {retry_wait}초 후 재시도...")
                time.sleep(retry_wait)
            else:
                print(f"  [{game_idx}] 연결 실패, 재시도 초과: {e}")
                return None

    res.encoding = "utf-8"
    soup = BeautifulSoup(res.text, "html.parser")

    game_name = soup.find("dl", class_="game_name")
    if not game_name:
        return None
    dt_el = game_name.find("dt")
    dd_el = game_name.find("dd")
    tournament = dt_el.get_text(strip=True) if dt_el else None

    dd_parts = [p.strip() for p in dd_el.get_text(strip=True).split("/")] if dd_el else []
    date_time = dd_parts[0] if len(dd_parts) > 0 else ""
    stadium = dd_parts[1] if len(dd_parts) > 1 else ""
    league = dd_parts[2] if len(dd_parts) > 2 else ""
    date_str, _, time_str = date_time.partition(" ")

    hero = soup.find("div", class_="hero_sum")
    if not hero:
        return None

    btn = hero.find("div", class_="btn_result")
    label = btn.get_text(strip=True) if btn else ""

    team1_dl = hero.select_one(".team1 .team")
    team2_dl = hero.select_one(".team2 .team")

    def _team_info(dl):
        if not dl:
            return None, "", ""
        name_el = dl.select_one("dt .txt")
        score_el = dl.select_one("dd.score")
        result_el = dl.select_one("dd.result")
        name = name_el.get_text(strip=True) if name_el else None
        score = score_el.get_text(strip=True) if score_el else ""
        result = result_el.get_text(strip=True) if result_el else ""
        return name, score, result

    team1, score1, result1 = _team_info(team1_dl)
    team2, score2, result2 = _team_info(team2_dl)

    return {
        "game_id": game_idx,
        "특이사항": label,
        "date": date_str,
        "time": time_str,
        "stadium": stadium,
        "league": league,
        "tournament": tournament,
        "team1": team1, "score1": score1, "result1": result1,
        "team2": team2, "score2": score2, "result2": result2,
    }


def crawl_special_games(months, year=YEAR):
    """캘린더에 "정상 종료"로 보이는 game_id 중, 우리가 원본 스코어시트로 갖고
    있지 않은 것만 골라 box_score로 상세를 확인해서 special_games_{YEAR}.xlsx로
    저장한다. 재실행해도 매번 "사이트 vs 우리 파일" 차집합을 새로 구해서 그중
    아직 저장 안 된 game_id만 새로 조회한다(이미 저장된 건 재사용)."""
    print("=== 원본 스코어시트에서 이미 확인된 game_id 집계 ===")
    known_ids = load_known_game_ids(YEAR_DIR, year)
    print(f"  raw 스코어시트 game_id {len(known_ids)}개\n")

    site_ids = set()
    for month in months:
        print(f"=== {year}년 {month}월 캘린더 조회 ===")
        ids = fetch_calendar_completed_ids(year, month)
        print(f"  정상 종료로 보이는 경기 {len(ids)}개")
        site_ids.update(ids)
        time.sleep(random.uniform(2, 3))

    missing_ids = sorted(site_ids - known_ids)
    print(f"\n사이트엔 있는데 우리 원본 파일엔 없는 game_id: {len(missing_ids)}개 -> {missing_ids}\n")

    existing_rows = []
    if os.path.exists(SPECIAL_OUT):
        prev_df = pd.read_excel(SPECIAL_OUT, dtype={"game_id": "Int64"})
        already = set(prev_df["game_id"].dropna().astype(int).tolist())
        existing_rows = prev_df.to_dict("records")
        missing_ids = [gid for gid in missing_ids if gid not in already]
        print(f"기존 special_games 파일에 {len(already)}행 있음 — 새로 조회할 game_id {len(missing_ids)}개\n")

    rows = list(existing_rows)
    suspended = []
    for i, gid in enumerate(missing_ids, 1):
        detail = fetch_special_result(gid)
        if detail and "일시정지" in str(detail["특이사항"]):
            # 일시정지(서스펜디드) 경기는 몰수·추첨승과 성격이 다르다 — 재개되면 진짜
            # 기록지가 올라오는 경기다. 여기서 결과만 확정해 넣으면 스트림릿 제조가
            # season_full 에 "선수 기록 없는 경기"로 굳혀버리고, 이 파일은 한 번 들어간
            # game_id 를 다시 조회하지 않아서 재개 뒤에도 스스로 낫지 않는다.
            # 스코어보드는 "경기 일정 크롤링.ipynb" 가 score_box_{YEAR}.xlsx 로 따로 받는다.
            suspended.append(detail)
            print(
                f"  [{i}/{len(missing_ids)}] {gid} {detail['team1']} {detail['score1']}:{detail['score2']} "
                f"{detail['team2']} · 일시정지경기 → special_games 에 넣지 않음(score_box 담당)"
            )
        elif detail:
            rows.append(detail)
            tag = detail["특이사항"] or "(라벨 없음 — 확인 필요)"
            print(
                f"  [{i}/{len(missing_ids)}] {gid} {detail['team1']} {detail['score1']}:{detail['score2']} "
                f"{detail['team2']} · {tag}"
            )
        else:
            print(f"  [{i}/{len(missing_ids)}] {gid} 조회 실패")
        time.sleep(random.uniform(9, 11))

    special_df = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(SPECIAL_OUT), exist_ok=True)
    special_df.to_excel(SPECIAL_OUT, index=False)
    print(f"\n완료: {SPECIAL_OUT} ({len(special_df)}행, 신규 {len(missing_ids)}행)")

    if suspended:
        print(f"\n일시정지 경기 {len(suspended)}건 — 재개 뒤 기록지가 올라오면 정상 경기로 처리됩니다:")
        for d in suspended:
            print(f"  {d['game_id']} {d['team1']} {d['score1']}:{d['score2']} {d['team2']} ({d['date']})")

    no_label = special_df[special_df["특이사항"] == ""] if not special_df.empty else special_df
    if len(no_label):
        print(f"\n⚠ '.btn_result' 라벨이 없는 game_id {len(no_label)}개 — 몰수/추첨이 아니라")
        print("  그냥 아직 원본 크롤러가 안 돈 정상 경기일 수 있습니다. 직접 확인해보세요:")
        print(no_label[["game_id", "team1", "score1", "team2", "score2", "date"]].to_string(index=False))

    return special_df


# ==================== 실행 ====================
# 시즌 시작 월부터 현재 월까지 훑는다.
special_df = crawl_special_games(months=[3, 4, 5, 6, 7])